# 01 — Téléchargement et inspection des données

**Objectif :** localiser les fichiers Kaggle et inspecter leur schéma sans exposer de données.
En leur absence, créer un petit corpus synthétique afin de tester tout le pipeline.

**Entrées :** `data/raw`.  
**Sorties :** `reports/data_inventory.json` et, en fallback, `data/processed/demo_source/`.  
**Dépendance :** notebook 00.  
**Temps estimé :** moins d'une minute hors téléchargement Kaggle.  
**Ressources :** CPU uniquement.

In [ ]:
from __future__ import annotations

import json
import sys
import zipfile
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Racine du projet introuvable.")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_loader import discover_competition_files, read_csv_as_strings

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
REPORTS_DIR = ROOT / "reports"
DEMO_DIR = PROCESSED_DIR / "demo_source"
for directory in (RAW_DIR, PROCESSED_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

## Extraction locale des archives déjà téléchargées

In [ ]:
for archive in RAW_DIR.glob("*.zip"):
    destination = RAW_DIR / archive.stem
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(destination)
    print(f"Archive extraite : {archive.name} → {destination.relative_to(ROOT)}")

competition = discover_competition_files(RAW_DIR)
print("Données Kaggle disponibles :", competition is not None)

Si les fichiers sont absents, exécuter dans PowerShell :

```powershell
.\.venv\Scripts\kaggle.exe auth login
.\.venv\Scripts\kaggle.exe competitions download `
  -c acm-sf-chapter-hackathon-small -p data\raw
```

Il peut être nécessaire d'accepter auparavant les règles sur la page du challenge.

## Corpus synthétique de secours

In [ ]:
products = [
    ("1001", "Halo Infinite", "Shooter", "Science-fiction shooter with Master Chief"),
    ("1002", "Forza Horizon 5", "Racing", "Open world car racing in Mexico"),
    ("1003", "Minecraft", "Sandbox", "Creative block building and survival game"),
    ("1004", "Call of Duty", "Shooter", "Military first person action shooter"),
    ("1005", "Grand Theft Auto V", "Action", "Open world crime action adventure"),
    ("1006", "EA Sports FC", "Sports", "Football and soccer simulation"),
    ("1007", "Gears 5", "Shooter", "Third person science-fiction shooter"),
    ("1008", "Sea of Thieves", "Adventure", "Cooperative pirate adventure"),
    ("1009", "Microsoft Flight Simulator", "Simulation", "Realistic airplane simulator"),
    ("1010", "Cyberpunk 2077", "RPG", "Futuristic open world role playing game"),
]

query_targets = {
    "halo": "1001",
    "halo game": "1001",
    "master chief": "1001",
    "forza": "1002",
    "forza horizon": "1002",
    "car racing": "1002",
    "mine craft": "1003",
    "minecraft xbox": "1003",
    "call duty": "1004",
    "call of duty": "1004",
    "gta 5": "1005",
    "grand theft auto": "1005",
    "football": "1006",
    "soccer game": "1006",
    "gears": "1007",
    "pirate game": "1008",
    "flight simulator": "1009",
    "cyber punk": "1010",
}

if competition is None:
    DEMO_DIR.mkdir(parents=True, exist_ok=True)
    product_frame = pd.DataFrame(products, columns=["sku", "title", "category", "description"])
    rows: list[dict[str, str]] = []
    for repeat in range(4):
        for index, (query, sku) in enumerate(query_targets.items()):
            category = product_frame.loc[product_frame["sku"] == sku, "category"].iloc[0]
            rows.append(
                {
                    "user": f"demo-{repeat:02d}-{index:02d}",
                    "sku": sku,
                    "category": category,
                    "query": query,
                    "click_time": f"2012-08-{repeat + 1:02d} 12:{index:02d}:30",
                    "query_time": f"2012-08-{repeat + 1:02d} 12:{index:02d}:00",
                }
            )
    train_frame = pd.DataFrame(rows)
    test_frame = pd.DataFrame(
        {
            "user": ["demo-test-1", "demo-test-2", "demo-test-3"],
            "category": ["Shooter", "Racing", "Adventure"],
            "query": ["halo xbox", "fast car game", "pirates"],
            "query_time": ["2012-09-01 10:00:00"] * 3,
        }
    )
    train_frame.to_csv(DEMO_DIR / "train.csv", index=False)
    test_frame.to_csv(DEMO_DIR / "test.csv", index=False)
    product_frame.to_csv(DEMO_DIR / "products.csv", index=False)
    source = "synthetic-demo"
else:
    train_frame = read_csv_as_strings(competition.train)
    test_frame = read_csv_as_strings(competition.test)
    source = "kaggle"

print(f"Source active : {source}")
print(f"Train : {train_frame.shape}; Test : {test_frame.shape}")
display(train_frame.head(3))

## Contrôles de schéma et inventaire

In [ ]:
required_train = {"sku", "query"}
required_test = {"query"}
assert required_train.issubset(train_frame.columns), train_frame.columns.tolist()
assert required_test.issubset(test_frame.columns), test_frame.columns.tolist()

inventory = {
    "source": source,
    "raw_files": sorted(path.name for path in RAW_DIR.rglob("*") if path.is_file()),
    "train_rows": int(len(train_frame)),
    "test_rows": int(len(test_frame)),
    "train_columns": train_frame.columns.tolist(),
    "test_columns": test_frame.columns.tolist(),
    "expected_kaggle_files": [
        "train.csv",
        "test.csv",
        "small_product_data.xml",
        "popular_skus.csv",
        "popular_skus.py",
    ],
}

(REPORTS_DIR / "data_inventory.json").write_text(
    json.dumps(inventory, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(json.dumps(inventory, indent=2, ensure_ascii=False))

## Conclusion

Les notebooks suivants privilégient automatiquement les données Kaggle lorsqu'elles sont
présentes. Le corpus synthétique ne sert qu'aux tests reproductibles et à l'intégration continue.